In [4]:
pip install SpeechRecognition pyttsx3 pyaudio


  Using cached pyttsx3-2.99-py3-none-any.whl.metadata (6.2 kB)
  Using cached pypiwin32-223-py3-none-any.whl.metadata (236 bytes)
   ---------------------------------------- 0.0/32.8 MB ? eta -:--:--
   ---------------------------------------- 0.0/32.8 MB ? eta -:--:--
   ---------------------------------------- 0.0/32.8 MB ? eta -:--:--
   ---------------------------------------- 0.0/32.8 MB ? eta -:--:--
   ---------------------------------------- 0.3/32.8 MB ? eta -:--:--
   ---------------------------------------- 0.3/32.8 MB ? eta -:--:--
   ---------------------------------------- 0.3/32.8 MB ? eta -:--:--
    --------------------------------------- 0.5/32.8 MB 430.4 kB/s eta 0:01:16
    --------------------------------------- 0.5/32.8 MB 430.4 kB/s eta 0:01:16
    --------------------------------------- 0.5/32.8 MB 430.4 kB/s eta 0:01:16
    --------------------------------------- 0.8/32.8 MB 430.1 kB/s eta 0:01:15
    --------------------------------------- 0.8/32.8 MB 430.1 k

In [2]:
!pip install transformers torch

In [3]:
from transformers import AutoModelForCausalLM, AutoTokenizer
import torch

# Load pretrained DialoGPT model
tokenizer = AutoTokenizer.from_pretrained("microsoft/DialoGPT-medium")
model = AutoModelForCausalLM.from_pretrained("microsoft/DialoGPT-medium")

print("Chatbot Ready! Type 'quit' to stop.")

chat_history_ids = None

while True:

    user_input = input("You: ")

    if user_input.lower() == "quit":
        print("Chatbot: Goodbye!")
        break

    # Encode user input
    new_input_ids = tokenizer.encode(
        user_input + tokenizer.eos_token,
        return_tensors='pt'
    )

    # Append chat history
    bot_input_ids = (
        torch.cat([chat_history_ids, new_input_ids], dim=-1)
        if chat_history_ids is not None
        else new_input_ids
    )

    # Generate response
    chat_history_ids = model.generate(
        bot_input_ids,
        max_length=1000,
        pad_token_id=tokenizer.eos_token_id,
        do_sample=True,
        top_k=50,
        top_p=0.95,
        temperature=0.7
    )

    # Decode response
    response = tokenizer.decode(
        chat_history_ids[:, bot_input_ids.shape[-1]:][0],
        skip_special_tokens=True
    )

    print("Chatbot:", response)

Xet Storage is enabled for this repo, but the 'hf_xet' package is not installed. Falling back to regular HTTP download. For better performance, install the package with: `pip install huggingface_hub[hf_xet]` or `pip install hf_xet`
C:\Users\anshu\cv_dl_env\lib\site-packages\torch\_utils.py:776: UserWarning: TypedStorage is deprecated. It will be removed in the future and UntypedStorage will be the only storage class. This should only matter to you if you are using storages directly.  To access UntypedStorage directly, use tensor.untyped_storage() instead of tensor.storage()
  return self.fget.__get__(instance, owner)()


Chatbot Ready! Type 'quit' to stop.


Xet Storage is enabled for this repo, but the 'hf_xet' package is not installed. Falling back to regular HTTP download. For better performance, install the package with: `pip install huggingface_hub[hf_xet]` or `pip install hf_xet`


You: hey


The attention mask is not set and cannot be inferred from input because pad token is same as eos token. As a consequence, you may observe unexpected behavior. Please pass your input's `attention_mask` to obtain reliable results.


Chatbot: hey u
You: how are you?
Chatbot: pretty good, thanks for asking
You: what is python programming?
Chatbot: It's a language where you write programs and write your own code.
You: what is ML
Chatbot: A language where you use algorithms and use the logic to make your code.
You: my name is anshul.
Chatbot: I just finished writing my last paper in that language!
You: quit
Chatbot: Goodbye!


In [1]:
from transformers import AutoModelForCausalLM, AutoTokenizer
import torch
import speech_recognition as sr
import pyttsx3

# Load DialoGPT
tokenizer = AutoTokenizer.from_pretrained("microsoft/DialoGPT-medium")
model = AutoModelForCausalLM.from_pretrained("microsoft/DialoGPT-medium")

# Text-to-Speech Engine
engine = pyttsx3.init()
engine.setProperty('rate', 170)

# Speech Recognizer
recognizer = sr.Recognizer()

chat_history_ids = None

print("Voice Chatbot Ready!")
print("Say 'stop' to stop.")

while True:

    try:
        with sr.Microphone() as source:
            print("\nListening...")
            recognizer.adjust_for_ambient_noise(source, duration=1)

            audio = recognizer.listen(source)

            user_input = recognizer.recognize_google(audio)

            print("You:", user_input)

    except Exception as e:
        print("Could not understand audio")
        continue

    # Exit
    if user_input.lower() == "stop":
        print("Chatbot: Goodbye!")
        engine.say("Goodbye")
        engine.runAndWait()
        break

    # Encode Input
    new_input_ids = tokenizer.encode(
        user_input + tokenizer.eos_token,
        return_tensors='pt'
    )

    # Add Conversation History
    bot_input_ids = (
        torch.cat([chat_history_ids, new_input_ids], dim=-1)
        if chat_history_ids is not None
        else new_input_ids
    )

    # Generate Response
    chat_history_ids = model.generate(
        bot_input_ids,
        max_length=1000,
        pad_token_id=tokenizer.eos_token_id,
        do_sample=True,
        top_k=50,
        top_p=0.95,
        temperature=0.7
    )

    # Decode Response
    response = tokenizer.decode(
        chat_history_ids[:, bot_input_ids.shape[-1]:][0],
        skip_special_tokens=True
    )

    print("Chatbot:", response)

    # Speak Response
    engine.say(response)
    engine.runAndWait()

C:\Users\anshu\cv_dl_env\lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
C:\Users\anshu\cv_dl_env\lib\site-packages\torchvision\datapoints\__init__.py:12: UserWarning: The torchvision.datapoints and torchvision.transforms.v2 namespaces are still Beta. While we do not expect major breaking changes, some APIs may still change according to user feedback. Please submit any feedback you may have in this issue: https://github.com/pytorch/vision/issues/6753, and you can also check out https://github.com/pytorch/vision/issues/7319 to learn more about the APIs that we suspect might involve future changes. You can silence this warning by calling torchvision.disable_beta_transforms_warning().
  warnings.warn(_BETA_TRANSFORMS_WARNING)
C:\Users\anshu\cv_dl_env\lib\site-packages\torchvision\transforms\v2\__init__.py:54: UserWar

Voice Chatbot Ready!
Say 'stop' to stop.

Listening...
You: hey how are you


The attention mask is not set and cannot be inferred from input because pad token is same as eos token. As a consequence, you may observe unexpected behavior. Please pass your input's `attention_mask` to obtain reliable results.


Chatbot: Good! How are you?

Listening...
Could not understand audio

Listening...
You: I'm bored can you tell me what is machine learning
Chatbot: Yeah sure

Listening...
You: what do you mean by machine learning
Chatbot: How to make more

Listening...
Could not understand audio

Listening...
You: stop
Chatbot: Goodbye!
